# Working with APIs & Real-World Data

See [Lessons/18-WorkingWithAPIs.md](../Lessons/18-WorkingWithAPIs.md) for the full lesson. No API key needed — this notebook uses the free [Open-Meteo](https://open-meteo.com/) weather API.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

## 1. Current weather for a city

Swap in your own latitude/longitude — these are Austin, TX.

In [ ]:
lat, lon = 30.27, -97.74

response = requests.get(
    'https://api.open-meteo.com/v1/forecast',
    params={'latitude': lat, 'longitude': lon, 'current': 'temperature_2m,wind_speed_10m'}
)
response.raise_for_status()
current = response.json()
current

## 2. Pull the `current` section into a DataFrame

In [ ]:
current_df = pd.json_normalize(current['current'])
current_df

## 3. 7-day forecast

Now request `daily` data and turn the list-of-values response into one row per day.

In [ ]:
response = requests.get(
    'https://api.open-meteo.com/v1/forecast',
    params={
        'latitude': lat,
        'longitude': lon,
        'daily': 'temperature_2m_max,temperature_2m_min',
        'timezone': 'auto',
    }
)
response.raise_for_status()
forecast = response.json()

# 'daily' here is a dict of parallel lists (one value per day) -- build a DataFrame directly from it
daily_df = pd.DataFrame(forecast['daily'])
daily_df

## 4. Answer with code

In [ ]:
# Highest forecasted max temperature in the next 7 days
daily_df['temperature_2m_max'].max()

In [ ]:
# Average forecasted min temperature
daily_df['temperature_2m_min'].mean()

In [ ]:
plt.plot(daily_df['time'], daily_df['temperature_2m_max'], label='Max')
plt.plot(daily_df['time'], daily_df['temperature_2m_min'], label='Min')
plt.xticks(rotation=45)
plt.ylabel('Temperature (°C)')
plt.title('7-Day Forecast')
plt.legend()
plt.tight_layout()
plt.show()

## Stretch Challenges
- Wrap the request in a function that checks `response.status_code` and raises a clear error if it failed
- Find a second no-key public API (see the list linked in the lesson) and pull one dataset from it